# SAR Automatic Target Recognition — Full Pipeline

**Environment**: Vast.ai GPU instance  
**Dataset**: MSTAR (10 vehicle classes, ~1000 SAR images)  
**Model**: CNN + GNN few-shot open-set recognition  

Run cells top-to-bottom. Each section is self-contained and can be re-run independently.

---

## 0. GPU Check

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — check instance type')

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. Environment Setup

In [ ]:
# Install all required packages
# torch / torchvision are usually pre-installed on Vast.ai PyTorch templates
# If not, uncomment the torch line below

!pip install -q \
    gdown \
    scikit-learn \
    scikit-image \
    matplotlib \
    Pillow \
    numpy

# Uncomment if torch is missing:
# !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Verify all imports work
import torch, torchvision, numpy, PIL, sklearn, matplotlib
print('All packages OK')
print(f'  torch       {torch.__version__}')
print(f'  torchvision {torchvision.__version__}')
print(f'  numpy       {numpy.__version__}')

## 2. Clone / Upload Project Code

Choose **one** of the two options below depending on how your code is stored.

In [ ]:
# ── OPTION A: Clone from GitHub (recommended) ──────────────────────────────
# Replace the URL with your actual repository URL

import os

REPO_URL = 'https://github.com/YOUR_USERNAME/SAR_Dr.Ahmed.git'   # <-- change this
PROJECT_DIR = '/workspace/SAR_Dr.Ahmed'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print('Directory already exists — pulling latest changes')
    !git -C {PROJECT_DIR} pull

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── OPTION B: Download zipped code from Google Drive ───────────────────────
# Use this if you uploaded the project as a .zip to your Drive
# Leave this cell unrun if you used Option A

import os, gdown, zipfile

CODE_GDRIVE_ID  = 'PASTE_YOUR_CODE_ZIP_FILE_ID_HERE'   # <-- change this
CODE_ZIP        = '/workspace/sar_code.zip'
PROJECT_DIR     = '/workspace/SAR_Dr.Ahmed'

if not os.path.exists(PROJECT_DIR):
    gdown.download(id=CODE_GDRIVE_ID, output=CODE_ZIP, quiet=False)
    with zipfile.ZipFile(CODE_ZIP, 'r') as z:
        z.extractall('/workspace')
    print('Code extracted to', PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

## 3. Download MSTAR Dataset from Google Drive

### How to get your Google Drive file ID

1. Upload `mstar_sampled_data.zip` (or the full MSTAR folder) to your Google Drive
2. Right-click the file → **Share** → set to **Anyone with the link**
3. Copy the share link — it looks like:  
   `https://drive.google.com/file/d/1aBcDeFgHiJkLmNoPqRsTuVwXyZ/view`
4. The file ID is the long string between `/d/` and `/view`:  
   `1aBcDeFgHiJkLmNoPqRsTuVwXyZ`
5. Paste it into `DATASET_GDRIVE_ID` below

In [ ]:
import os, gdown, zipfile

# ── Configuration ──────────────────────────────────────────────────────────
DATASET_GDRIVE_ID = 'PASTE_YOUR_DATASET_FILE_ID_HERE'   # <-- change this
DATASET_ZIP       = '/workspace/mstar_sampled_data.zip'
DATASET_DIR       = os.path.join(os.getcwd(), 'mstar_sampled_data')
# ───────────────────────────────────────────────────────────────────────────

if os.path.exists(DATASET_DIR):
    print(f'Dataset directory already exists: {DATASET_DIR}')
    print('Delete it and re-run this cell to force a fresh download.')
else:
    print('Downloading dataset from Google Drive...')
    gdown.download(id=DATASET_GDRIVE_ID, output=DATASET_ZIP, quiet=False)

    print('Extracting...')
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(os.getcwd())

    os.remove(DATASET_ZIP)
    print('Done.')

In [ ]:
# Verify dataset structure
import os

DATASET_DIR = os.path.join(os.getcwd(), 'mstar_sampled_data')

if not os.path.exists(DATASET_DIR):
    raise FileNotFoundError(f'Dataset not found at {DATASET_DIR} — run the download cell first')

classes = sorted(os.listdir(DATASET_DIR))
print(f'Found {len(classes)} classes:\n')
total = 0
for cls in classes:
    cls_path = os.path.join(DATASET_DIR, cls)
    if not os.path.isdir(cls_path):
        continue
    n = len(os.listdir(cls_path))
    total += n
    marker = '  [UNSEEN - held out]' if cls == 'T72' else ''
    print(f'  {cls:<15} {n:>4} images{marker}')

print(f'\nTotal images : {total}')
assert 'T72' in classes, 'T72 class not found — check dataset structure'
print('\nDataset structure OK')

In [ ]:
# Preview a few images from each class
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os, random

DATASET_DIR = os.path.join(os.getcwd(), 'mstar_sampled_data')
classes = sorted([c for c in os.listdir(DATASET_DIR)
                  if os.path.isdir(os.path.join(DATASET_DIR, c))])

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('MSTAR Dataset — One sample per class', fontsize=13)

for ax, cls in zip(axes.flat, classes):
    cls_dir = os.path.join(DATASET_DIR, cls)
    img_file = random.choice(os.listdir(cls_dir))
    img = mpimg.imread(os.path.join(cls_dir, img_file))
    ax.imshow(img, cmap='gray')
    title = cls + (' [UNSEEN]' if cls == 'T72' else '')
    ax.set_title(title, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('results/dataset_preview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to results/dataset_preview.png')

## 4. Create Output Directories

In [ ]:
import os
for d in ['model', 'log', 'results', 'gan_output', 'model/gan']:
    os.makedirs(d, exist_ok=True)
print('Output directories ready')

## 5. Shape Smoke Test

Verifies CNN and GNN produce the correct tensor shapes before committing to a long training run.

In [ ]:
import torch
from cnn import EmbeddingCNN
from gnn import GNN_module

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running smoke test on: {device}\n')

# CNN: (batch=4, channels=1, H=100, W=100) -> (batch, 64)
cnn = EmbeddingCNN(image_size=100, cnn_feature_size=64,
                   cnn_hidden_dim=32, cnn_num_layers=4).to(device)
x = torch.rand(4, 1, 100, 100, device=device)
out = cnn(x)
assert out.shape == (4, 64), f'CNN shape error: {out.shape}'
print(f'CNN output shape  : {list(out.shape)}  ✓')

# GNN: 3-way 20-shot, batch=8
nway, shots, batch = 3, 20, 8
D = 64 + nway + 1          # feature dim = CNN embed + one-hot label
N = nway * shots + 1       # nodes = support + 1 query
nodes = torch.rand(batch, N, D, device=device)
gnn = GNN_module(nway=nway, input_dim=D, hidden_dim=16).to(device)
logits = gnn(nodes)
assert logits.shape == (batch, nway + 1), f'GNN shape error: {logits.shape}'
print(f'GNN output shape  : {list(logits.shape)}  ✓  (batch × C+1 logits)')

# Data loader
from data import self_DataLoader
dl = self_DataLoader(root='mstar_sampled_data', nway=3,
                     unseen_class='T72', unseen_ratio=1.0)
batch_data = dl.load_tr_batch(batch_size=4, nway=3, num_shots=5)
print(f'DataLoader batch  : {[list(b.shape) for b in batch_data[:4]]}  ✓')

print('\nAll smoke tests PASSED — safe to train')

---
## 6. Experiment 1 — Paper Baseline: FSL + GNN (3-way 20-shot)

Reproduces the reference paper configuration.  
**Target**: seen ≥ 95%, unseen ≥ 62%, overall ≥ 85%  
**Expected runtime**: ~40 min on a single GPU

In [ ]:
!python main.py \
    --data_root     mstar_sampled_data \
    --model_type    gnn \
    --nway          3 \
    --shots         20 \
    --unseen_class  T72 \
    --unseen_ratio  1.0 \
    --batch_size    8 \
    --lr            0.001 \
    --max_iteration 100000 \
    --log_interval  100 \
    --eval_interval 500 \
    --early_stop    10 \
    --eval_output   results \
    --save \
    --use_gpu       0

In [ ]:
# Read and display the last 30 lines of the training log
import glob, os

log_files = glob.glob('log/3way_20shot_gnn_/*_log.txt')
if log_files:
    with open(log_files[0]) as f:
        lines = f.readlines()
    print(''.join(lines[-30:]))
else:
    print('Log file not found — check log/ directory')

## 7. Experiment 2 — CNN Baseline (Closed-Set)

Trains a standard CNN classifier without few-shot learning to establish how bad things are with limited data.  
**Expected runtime**: ~10 min

In [ ]:
# Full-data CNN baseline (all 80 training images per class)
!python main.py \
    --data_root     mstar_sampled_data \
    --model_type    cnn \
    --nway          9 \
    --shots         20 \
    --batch_size    8 \
    --lr            0.001 \
    --max_iteration 50000 \
    --log_interval  100 \
    --eval_interval 500 \
    --early_stop    10 \
    --affix         full \
    --eval_output   results \
    --save \
    --use_gpu       0

In [ ]:
# K=10 shot CNN baseline (only 10 images per class — simulating data scarcity)
!python main.py \
    --data_root     mstar_sampled_data \
    --model_type    cnn \
    --nway          9 \
    --shots         10 \
    --baseline_kshot \
    --batch_size    8 \
    --lr            0.001 \
    --max_iteration 50000 \
    --log_interval  100 \
    --eval_interval 500 \
    --early_stop    10 \
    --affix         k10 \
    --eval_output   results \
    --save \
    --use_gpu       0

## 8. Experiment 3 — 5-way and 7-way Configurations

Tests whether the GNN generalizes to more classes per episode.

In [ ]:
# 5-way 20-shot
!python main.py \
    --data_root     mstar_sampled_data \
    --model_type    gnn \
    --nway          5 \
    --shots         20 \
    --unseen_class  T72 \
    --unseen_ratio  1.0 \
    --batch_size    8 \
    --lr            0.001 \
    --max_iteration 100000 \
    --early_stop    10 \
    --eval_output   results \
    --save \
    --use_gpu       0

In [ ]:
# 7-way 20-shot
!python main.py \
    --data_root     mstar_sampled_data \
    --model_type    gnn \
    --nway          7 \
    --shots         20 \
    --unseen_class  T72 \
    --unseen_ratio  1.0 \
    --batch_size    8 \
    --lr            0.001 \
    --max_iteration 100000 \
    --early_stop    10 \
    --eval_output   results \
    --save \
    --use_gpu       0

## 9. Experiment 4 — GAN Training and Augmented FSL

### Step 1: Train the Conditional DCGAN

Generates 100 synthetic SAR images per vehicle class.  
**Expected runtime**: ~60 min for 200 epochs on GPU

In [ ]:
!python gan.py \
    --data_root   mstar_sampled_data \
    --output_dir  gan_output \
    --epochs      200 \
    --batch_size  32 \
    --noise_dim   100 \
    --img_size    100 \
    --lr_g        0.0002 \
    --lr_d        0.0002 \
    --n_generate  100 \
    --seed        42 \
    --gpu         0

In [ ]:
# Verify GAN output: count generated images per class
import os

gan_dir = 'gan_output'
if not os.path.exists(gan_dir):
    print('gan_output/ not found — run the GAN training cell first')
else:
    classes = sorted(os.listdir(gan_dir))
    print(f'GAN output — {gan_dir}/')
    for cls in classes:
        cls_path = os.path.join(gan_dir, cls)
        if os.path.isdir(cls_path):
            n = len(os.listdir(cls_path))
            status = '  ✓' if n >= 100 else f'  WARNING: only {n} images'
            print(f'  {cls:<15} {n:>4} images{status}')

In [ ]:
# Preview GAN-generated images side-by-side with real ones
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os, random

classes_to_show = ['2S1', 'BMP2', 'T62']
fig, axes = plt.subplots(len(classes_to_show), 4, figsize=(10, 7))
fig.suptitle('Real (left 2) vs GAN-generated (right 2)', fontsize=12)

for row, cls in enumerate(classes_to_show):
    real_dir = os.path.join('mstar_sampled_data', cls)
    fake_dir = os.path.join('gan_output', cls)

    for col, img_dir in enumerate([real_dir, real_dir, fake_dir, fake_dir]):
        if not os.path.exists(img_dir):
            axes[row, col].axis('off')
            continue
        img_file = random.choice(os.listdir(img_dir))
        img = mpimg.imread(os.path.join(img_dir, img_file))
        axes[row, col].imshow(img, cmap='gray')
        label = ('Real' if col < 2 else 'GAN') + f'\n{cls}'
        axes[row, col].set_title(label, fontsize=8)
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('results/gan_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to results/gan_comparison.png')

### Step 2: FSL + GNN with GAN Augmentation (K=10)

In [ ]:
!python main.py \
    --data_root      mstar_sampled_data \
    --model_type     gnn \
    --nway           3 \
    --shots          10 \
    --unseen_class   T72 \
    --gan_augment \
    --gan_output_dir gan_output \
    --batch_size     8 \
    --lr             0.001 \
    --max_iteration  100000 \
    --early_stop     10 \
    --affix          gan_k10 \
    --eval_output    results \
    --save \
    --use_gpu        0

## 10. Experiment 5 — Physics-Informed Regularization

Adds the intra-class embedding variance penalty to the loss.

In [ ]:
# With physics regularization (lambda = 0.05)
!python main.py \
    --data_root      mstar_sampled_data \
    --model_type     gnn \
    --nway           3 \
    --shots          20 \
    --unseen_class   T72 \
    --physics_lambda 0.05 \
    --batch_size     8 \
    --lr             0.001 \
    --max_iteration  100000 \
    --early_stop     10 \
    --affix          physics \
    --eval_output    results \
    --save \
    --use_gpu        0

## 11. Experiment 6 — Rotation and Speckle Noise Invariance

In [ ]:
# With rotation + speckle augmentation during training
!python main.py \
    --data_root       mstar_sampled_data \
    --model_type      gnn \
    --nway            3 \
    --shots           20 \
    --unseen_class    T72 \
    --augment_rotation \
    --augment_speckle \
    --speckle_sigma   0.1 \
    --batch_size      8 \
    --lr              0.001 \
    --max_iteration   100000 \
    --early_stop      10 \
    --affix           augmented \
    --eval_output     results \
    --save \
    --use_gpu         0

## 12. Experiment 7 — Full Pipeline (Everything Together)

GAN augmentation + physics regularization + rotation + speckle noise, at K=10.

In [ ]:
!python main.py \
    --data_root       mstar_sampled_data \
    --model_type      gnn \
    --nway            3 \
    --shots           10 \
    --unseen_class    T72 \
    --gan_augment \
    --gan_output_dir  gan_output \
    --physics_lambda  0.05 \
    --augment_rotation \
    --augment_speckle \
    --speckle_sigma   0.1 \
    --batch_size      8 \
    --lr              0.001 \
    --max_iteration   100000 \
    --early_stop      10 \
    --affix           full_pipeline \
    --eval_output     results \
    --save \
    --use_gpu         0

## 13. Evaluate a Saved Checkpoint (No Re-Training)

Use `--eval_only` to load any checkpoint and run a fresh 5000-task evaluation.

In [ ]:
# Evaluate the paper-baseline checkpoint from Experiment 1
!python main.py \
    --data_root    mstar_sampled_data \
    --model_type   gnn \
    --nway         3 \
    --shots        20 \
    --unseen_class T72 \
    --load \
    --load_dir     model/3way_20shot_gnn_ \
    --eval_only \
    --eval_output  results \
    --use_gpu      0

## 14. Results: Comparison Table Across All Experiments

In [ ]:
import json, glob, os
from evaluate import print_comparison_table

report_files = sorted(glob.glob('results/*.json'))

if not report_files:
    print('No result JSON files found yet — run training experiments first')
else:
    print(f'Found {len(report_files)} report(s):\n')
    for f in report_files:
        print(' ', f)
    print()
    table = print_comparison_table(report_files)

    # Save to markdown file
    with open('results/comparison_table.md', 'w') as f:
        f.write('# Results Comparison\n\n')
        f.write(table)
    print('\nSaved to results/comparison_table.md')

In [ ]:
# Bar chart: Overall accuracy across experiments
import json, glob
import matplotlib.pyplot as plt

report_files = sorted(glob.glob('results/*.json'))

if report_files:
    configs, overall, seen, unseen = [], [], [], []

    for path in report_files:
        with open(path) as f:
            d = json.load(f)
        label = d.get('config', os.path.basename(path).replace('.json',''))
        configs.append(label)
        overall.append(d.get('overall_accuracy', 0))
        seen.append(d.get('seen_accuracy', 0))
        unseen.append(d.get('unseen_accuracy', 0))

    x = range(len(configs))
    width = 0.25

    fig, ax = plt.subplots(figsize=(max(8, len(configs) * 1.8), 5))
    ax.bar([i - width for i in x], overall, width, label='Overall', color='steelblue')
    ax.bar([i         for i in x], seen,    width, label='Seen',    color='seagreen')
    ax.bar([i + width for i in x], unseen,  width, label='Unseen',  color='tomato')

    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Accuracy by Experiment and Split')
    ax.set_xticks(list(x))
    ax.set_xticklabels(configs, rotation=30, ha='right', fontsize=8)
    ax.axhline(95, linestyle='--', color='seagreen', alpha=0.4, label='Seen target (95%)')
    ax.axhline(62, linestyle='--', color='tomato',   alpha=0.4, label='Unseen target (62%)')
    ax.legend()
    ax.set_ylim(0, 110)
    plt.tight_layout()
    plt.savefig('results/accuracy_chart.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved to results/accuracy_chart.png')
else:
    print('No JSON reports found yet')

In [ ]:
# Confusion matrix for the paper-baseline experiment
import json, glob
import numpy as np
import matplotlib.pyplot as plt

baseline_reports = glob.glob('results/3way_20shot_gnn_*.json')

if baseline_reports:
    with open(baseline_reports[-1]) as f:
        d = json.load(f)

    cm = np.array(d['confusion_matrix'])
    class_names = list(d['per_class'].keys())

    # Normalize
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(class_names, fontsize=8)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('Confusion Matrix — 3-way 20-shot GNN')

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cm_norm[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=7, color='white' if val > 0.5 else 'black')

    plt.tight_layout()
    plt.savefig('results/confusion_matrix.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved to results/confusion_matrix.png')
else:
    print('No baseline report found — run Experiment 1 first')

## 15. Save All Results to Google Drive

Uploads the `results/`, `model/`, and `log/` directories back to your Drive so nothing is lost when the instance is terminated.

In [ ]:
# Authenticate Google Drive
from google.colab import drive
drive.mount('/gdrive')

# NOTE: If running on Vast.ai (not Colab), use the gdown upload approach or rclone instead.
# See the rclone cell below.

In [ ]:
# ── Google Colab: copy to mounted Drive ────────────────────────────────────
import shutil, os

DRIVE_DEST = '/gdrive/MyDrive/SAR_results'   # <-- change destination folder
os.makedirs(DRIVE_DEST, exist_ok=True)

for folder in ['results', 'log', 'model']:
    src = os.path.join(os.getcwd(), folder)
    dst = os.path.join(DRIVE_DEST, folder)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Copied {folder}/ -> {dst}')

print('\nAll results saved to Google Drive')

In [ ]:
# ── Vast.ai (no Colab): zip results and download via browser ───────────────
# This creates a single zip you can download from the Jupyter file browser

import shutil, os, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
zip_name  = f'sar_results_{timestamp}'
zip_path  = f'/workspace/{zip_name}'

shutil.make_archive(zip_path, 'zip', os.getcwd(),
                    base_dir=None,
                    root_dir=os.getcwd())

final = zip_path + '.zip'
size_mb = os.path.getsize(final) / 1e6
print(f'Archive created: {final}  ({size_mb:.1f} MB)')
print('Download it from the Jupyter file browser on the left panel')

In [ ]:
# ── Vast.ai: Upload via rclone to Google Drive (most reliable for large files) ─
# Run once to configure rclone (interactive — do this in a terminal, not here)
# Then run this cell to sync

# First-time setup (run in terminal):
#   rclone config
#   -> choose Google Drive, name it 'gdrive'

GDRIVE_REMOTE_PATH = 'gdrive:SAR_results'   # <-- change folder name

!rclone copy results/ {GDRIVE_REMOTE_PATH}/results/ --progress
!rclone copy model/   {GDRIVE_REMOTE_PATH}/model/   --progress
!rclone copy log/     {GDRIVE_REMOTE_PATH}/log/     --progress
print('Upload complete')

---
## Quick Reference

| Cell | What it runs | Approximate GPU time |
|------|-------------|---------------------|
| §5  | Smoke test | < 1 min |
| §6  | FSL+GNN 3-way 20-shot (paper baseline) | ~40 min |
| §7  | CNN baseline (full + K=10) | ~10 min each |
| §8  | 5-way + 7-way GNN | ~40 min each |
| §9  | GAN training (200 epochs) | ~60 min |
| §9  | FSL+GNN + GAN augment K=10 | ~40 min |
| §10 | Physics loss ablation | ~40 min |
| §11 | Rotation + speckle augmentation | ~40 min |
| §12 | Full pipeline (all enhancements) | ~40 min |
| §13 | Eval-only (any checkpoint) | ~5 min |
| §14 | Comparison table + charts | < 1 min |

**Total for all experiments**: ~6 hours. Run §6 first to verify targets are met before committing to the rest.

### Key paths
```
model/3way_20shot_gnn_/model.pth   ← best checkpoint (Exp 1)
model/gan/generator.pt             ← trained GAN generator
results/*.json                     ← per-experiment metrics
results/comparison_table.md        ← printable results table
results/accuracy_chart.png         ← bar chart
results/confusion_matrix.png       ← confusion matrix (Exp 1)
log/*/train_log.txt                ← full training logs
```